# 07_joining

This notebook creates the joined contract master table used downstream for feature engineering, weak supervision, and gold-label evaluation.

Main tasks:
1. Load the cleaned source tables from the project data folders
2. Join supplier bridge, financials, ESG, news, stocks, and macro indexes
3. Create a simple manually maintained gold-label table
4. Join `gold_y` onto the final contract table
5. Save both:
   - joined contract table without gold labels
   - joined contract table with gold labels
   - manual gold label table

## Imports and project path setup

In [128]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
src_path = project_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("project_root:", project_root)
print("src_path:", src_path)

project_root: /Users/annita/Desktop/Thesis/Signal_Fusion_with_Meta-learners-
src_path: /Users/annita/Desktop/Thesis/Signal_Fusion_with_Meta-learners-/src


In [129]:
import sys
import os
sys.path.append(os.path.abspath("../src"))

import pandas as pd
import numpy as np

from master_thesis.data_utils import load_raw, save_processed

## File names

Update if we change raw files naming

In [130]:
RAW_FILES = {
    "contract_year": "contract_year_final.csv",
    "supplier_bridge": "dim_supplier.csv",
    "financials": "financials_clean.csv",
    "esg": "esg_yearly.csv",
    "news": "news.csv",
    "stocks": "stock_view.csv",      # update if your file is named stock_view.csv
    "indexes": "macro_indexs.csv",   # update if your file is named macro_indexes.csv
}

## Load core tables

In [131]:
df_contract_year = load_raw(RAW_FILES["contract_year"], low_memory=False)
df_suppliers_bridge = load_raw(RAW_FILES["supplier_bridge"], low_memory=False)
df_financials_clean = load_raw(RAW_FILES["financials"], low_memory=False)
df_esg_yearly = load_raw(RAW_FILES["esg"], low_memory=False)
df_news = load_raw(RAW_FILES["news"], low_memory=False)
df_stocks = load_raw(RAW_FILES["stocks"], low_memory=False)
df_indexes = load_raw(RAW_FILES["indexes"], low_memory=False)

print("contract_year:", df_contract_year.shape)
print("supplier_bridge:", df_suppliers_bridge.shape)
print("financials:", df_financials_clean.shape)
print("esg:", df_esg_yearly.shape)
print("news:", df_news.shape)
print("stocks:", df_stocks.shape)
print("indexes:", df_indexes.shape)

contract_year: (9201, 34)
supplier_bridge: (1000, 51)
financials: (3529, 54)
esg: (2093, 16)
news: (835, 11)
stocks: (30304, 28)
indexes: (1635, 11)


## Helper functions

In [132]:
def clean_id_like_column(series: pd.Series) -> pd.Series:
    return (
        series.astype(str)
        .str.replace(r"\.0$", "", regex=True)
        .str.strip()
    )

def clean_country_name(value):
    if pd.isna(value):
        return pd.NA

    value = str(value).strip()

    mapping = {
        "RUSSIAN FEDERATION": "Russia",
        "UNITED STATES": "United States",
        "Novo Nordisk Saudi Arabia": "Saudi Arabia",
    }

    if value in mapping:
        return mapping[value]

    if value.isupper():
        return value.title()

    return value

def build_manual_gold_table(gold_contract_map: dict) -> pd.DataFrame:
    rows = []

    for department, labels in gold_contract_map.items():
        for contract_number in labels.get("yes", []):
            rows.append({
                "contract_number": str(contract_number),
                "gold_y": 1,
                "gold_department": department,
            })
        for contract_number in labels.get("no", []):
            rows.append({
                "contract_number": str(contract_number),
                "gold_y": 0,
                "gold_department": department,
            })

    df_gold = pd.DataFrame(rows).drop_duplicates(subset=["contract_number"]).copy()
    df_gold["contract_number"] = clean_id_like_column(df_gold["contract_number"])
    return df_gold

def validate_manual_gold_labels(
    df_gold_manual: pd.DataFrame,
    df_contracts: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    df_contracts_check = df_contracts.copy()
    df_contracts_check["contract_number"] = clean_id_like_column(df_contracts_check["contract_number"])

    df_matches = df_gold_manual.merge(
        df_contracts_check[
            ["contract_id", "contract_number", "contract_name", "department"]
        ].drop_duplicates(),
        on="contract_number",
        how="left",
    )

    df_valid = df_matches[df_matches["contract_id"].notna()].copy()
    df_missing = df_matches[df_matches["contract_id"].isna()].copy()

    df_summary = (
        df_valid.groupby("gold_y")
        .agg(
            n_rows=("contract_number", "size"),
            n_unique_contracts=("contract_number", "nunique"),
        )
        .reset_index()
        .sort_values("gold_y", ascending=False)
    )

    return df_matches, df_valid, df_missing

## Contract-year base table

In [133]:
df_contract = df_contract_year.copy()

df_contract["supplier_number"] = clean_id_like_column(df_contract["supplier_number"])
df_contract["contract_id"] = clean_id_like_column(df_contract["contract_id"])

df_contract["observation_year"] = pd.to_numeric(
    df_contract["observation_year"],
    errors="coerce",
).astype("Int64")

print("Shape:", df_contract.shape)
print("Unique contracts:", df_contract["contract_id"].nunique())
print("Duplicate contract-year rows:", df_contract.duplicated(["contract_id", "observation_year"]).sum())
display(df_contract.head())

Shape: (9201, 34)
Unique contracts: 2209
Duplicate contract-year rows: 0


,contract_id,contract_number,contract_name,contract_status,contract_owner_cost_centre,terminated,term_type,start_date,expiration_date,supplier_id,...,expiry_year,open_ended_contract,end_year,start_year_capped,observation_year,years_to_expiry,contract_age_years,expiry_pressure_bucket,department_from_cost_center,department
0,9675,9675,Bioreliance_Master_2018_MSA,published,7756.0,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,...,2027,0,2025,2018,2018,9,0,5y_plus,NaN,"Quality, Production Services & Supplies"
1,9675,9675,Bioreliance_Master_2018_MSA,published,7756.0,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,...,2027,0,2025,2018,2019,8,1,5y_plus,NaN,"Quality, Production Services & Supplies"
2,9675,9675,Bioreliance_Master_2018_MSA,published,7756.0,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,...,2027,0,2025,2018,2020,7,2,5y_plus,NaN,"Quality, Production Services & Supplies"
3,9675,9675,Bioreliance_Master_2018_MSA,published,7756.0,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,...,2027,0,2025,2018,2021,6,3,5y_plus,NaN,"Quality, Production Services & Supplies"
4,9675,9675,Bioreliance_Master_2018_MSA,published,7756.0,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,...,2027,0,2025,2018,2022,5,4,3_to_5y,NaN,"Quality, Production Services & Supplies"


## Supplier bridge join

In [134]:
df_bridge = df_suppliers_bridge.copy()

df_bridge["supplier_key"] = clean_id_like_column(df_bridge["supplier_key"])
df_bridge["moodys_bvd_id"] = df_bridge["moodys_bvd_id"].astype("string").str.strip()

df_bridge_slim = (
    df_bridge[["supplier_key", "moodys_bvd_id"]]
    .drop_duplicates()
    .copy()
)

print("Duplicate bridge supplier_key rows:", df_bridge_slim.duplicated(["supplier_key"]).sum())

Duplicate bridge supplier_key rows: 0


In [135]:
df_contract_bvd = (
    df_contract.merge(
        df_bridge_slim,
        left_on="supplier_number",
        right_on="supplier_key",
        how="left",
    )
    .drop(columns=["supplier_key"])
)

print("Rows:", len(df_contract_bvd))
print("Matched to BvD ID:", df_contract_bvd["moodys_bvd_id"].notna().sum())
print("Duplicate contract-year rows:", df_contract_bvd.duplicated(["contract_id", "observation_year"]).sum())

Rows: 9201
Matched to BvD ID: 4634
Duplicate contract-year rows: 0


## Financials join

In [136]:
df_fin = df_financials_clean.copy()

df_fin["moodys_bvd_id"] = df_fin["moodys_bvd_id"].astype("string").str.strip()
df_fin["Join_Year"] = pd.to_numeric(df_fin["Join_Year"], errors="coerce").astype("Int64")

fin_key_cols = ["moodys_bvd_id", "Join_Year"]
fin_other_cols = [c for c in df_fin.columns if c not in fin_key_cols]

df_fin_prefixed = df_fin.rename(columns={c: f"fin_{c}" for c in fin_other_cols})

print("Duplicate financial company-year:", df_fin.duplicated(["moodys_bvd_id", "Join_Year"]).sum())

Duplicate financial company-year: 0


In [137]:
df_contract_fin = (
    df_contract_bvd.merge(
        df_fin_prefixed,
        left_on=["moodys_bvd_id", "observation_year"],
        right_on=["moodys_bvd_id", "Join_Year"],
        how="left",
    )
    .drop(columns=["Join_Year"])
)

print("Rows:", len(df_contract_fin))
print("Financial matches:", df_contract_fin["fin_Operating_revenue_Turnover_th_DKK"].notna().sum())
print("Duplicate contract-year rows:", df_contract_fin.duplicated(["contract_id", "observation_year"]).sum())

Rows: 9201
Financial matches: 1281
Duplicate contract-year rows: 0


## ESG join

In [138]:
df_esg = df_esg_yearly.copy()

df_esg["moodys_bvd_id"] = df_esg["moodys_bvd_id"].astype("string").str.strip()
df_esg["Join_Year"] = pd.to_numeric(df_esg["Join_Year"], errors="coerce").astype("Int64")

esg_key_cols = ["moodys_bvd_id", "Join_Year"]
esg_other_cols = [c for c in df_esg.columns if c not in esg_key_cols]

df_esg_prefixed = df_esg.rename(columns={c: f"esg_{c}" for c in esg_other_cols})

df_contract_fin_esg = (
    df_contract_fin.merge(
        df_esg_prefixed,
        left_on=["moodys_bvd_id", "observation_year"],
        right_on=["moodys_bvd_id", "Join_Year"],
        how="left",
    )
    .drop(columns=["Join_Year"])
)

print("Rows:", len(df_contract_fin_esg))
print("ESG matches:", df_contract_fin_esg["esg_esg_overall"].notna().sum())
print("Duplicate contract-year rows:", df_contract_fin_esg.duplicated(["contract_id", "observation_year"]).sum())

Rows: 9201
ESG matches: 1012
Duplicate contract-year rows: 0


## News join

In [139]:
df_news_view = df_news.copy()

df_news_view["supplier_number"] = clean_id_like_column(df_news_view["supplier_number"])
df_news_view["Join_Year"] = pd.to_numeric(df_news_view["Join_Year"], errors="coerce").astype("Int64")

news_key_cols = ["supplier_number", "Join_Year"]
news_other_cols = [c for c in df_news_view.columns if c not in news_key_cols]

df_news_prefixed = df_news_view.rename(columns={c: f"news_{c}" for c in news_other_cols})

df_feature_matrix = (
    df_contract_fin_esg.merge(
        df_news_prefixed,
        left_on=["supplier_number", "observation_year"],
        right_on=["supplier_number", "Join_Year"],
        how="left",
    )
    .drop(columns=["Join_Year"])
)

print("Rows:", len(df_feature_matrix))
print("News matches:", df_feature_matrix["news_article_count"].notna().sum())
print("Duplicate contract-year rows:", df_feature_matrix.duplicated(["contract_id", "observation_year"]).sum())

Rows: 9201
News matches: 1185
Duplicate contract-year rows: 0


## Stocks join

In [140]:
df_stocks_clean = df_stocks.rename(
    columns={
        "BvD ID number": "moodys_bvd_id",
        "Join_Year": "observation_year",
    }
).copy()

df_stocks_clean = df_stocks_clean.drop(columns=["Unnamed: 0"], errors="ignore")

if "moodys_bvd_id" in df_stocks_clean.columns:
    df_stocks_clean["moodys_bvd_id"] = df_stocks_clean["moodys_bvd_id"].astype("string").str.strip()

if "observation_year" in df_stocks_clean.columns:
    df_stocks_clean["observation_year"] = pd.to_numeric(df_stocks_clean["observation_year"], errors="coerce").astype("Int64")

join_keys = ["moodys_bvd_id", "observation_year"]

stock_new_columns = [
    col for col in df_stocks_clean.columns
    if col not in df_feature_matrix.columns or col in join_keys
]

df_stocks_to_merge = df_stocks_clean[stock_new_columns].copy()

print("Duplicate stock rows on join keys:", df_stocks_to_merge.duplicated(subset=join_keys).sum())

Duplicate stock rows on join keys: 0


In [141]:
df_feature_matrix_enriched = df_feature_matrix.merge(
    df_stocks_to_merge,
    on=join_keys,
    how="left",
)

print("Original shape:", df_feature_matrix.shape)
print("Enriched shape:", df_feature_matrix_enriched.shape)

added_stock_columns = [c for c in df_feature_matrix_enriched.columns if c not in df_feature_matrix.columns]
print("Added stock columns:")
print(added_stock_columns)

Original shape: (9201, 110)
Enriched shape: (9201, 136)
Added stock columns:
['Company name Latin alphabet', 'year', 'avg_vol', 'std_vol', 'max_vol', 'min_vol', 'vol_stability_score', 'vol_shock_ratio', 'vol_trend_slope', 'avg_market_cap', 'market_cap_volatility', 'Risk level', 'supplier_sector', 'moodys_risk_rating', 'Price_trends_52_weeks_pct', 'market_beta_1y', 'Earnings_per_share_DKK', 'Book_value_per_share_DKK', 'Shares outstanding', 'Current_market_capitalisation_DKK', 'avg_shares_outstanding', 'Risk level_monthly_shares', 'avg_closing_price', 'price_volatility_score', 'price_trend_slope', 'Risk level_stock_closing_price']


## Macro index join

In [142]:
df_left = df_feature_matrix_enriched.copy()
df_right = df_indexes.copy()

df_right = df_right.drop(columns=["Unnamed: 0"], errors="ignore")
df_right = df_right.rename(columns={"Year": "observation_year"})

df_left["company_country_clean"] = df_left["company_country"].apply(clean_country_name)
df_right["Country_clean"] = df_right["Country"].astype(str).str.strip()
df_right = df_right.rename(columns={"Country_clean": "company_country_clean"})

df_right["observation_year"] = pd.to_numeric(df_right["observation_year"], errors="coerce").astype("Int64")

join_keys = ["company_country_clean", "observation_year"]

macro_cols_to_merge = [
    col for col in df_right.columns
    if col not in df_left.columns or col in join_keys
]

df_right_to_merge = df_right[macro_cols_to_merge].copy()

print("Duplicate macro rows on join keys:", df_right_to_merge.duplicated(subset=join_keys).sum())

Duplicate macro rows on join keys: 0


In [143]:
df_contract_final_no_spend = df_left.merge(
    df_right_to_merge,
    on=join_keys,
    how="left",
)

added_macro_columns = [c for c in df_contract_final_no_spend.columns if c not in df_feature_matrix_enriched.columns]

print("Added macro columns:")
print(added_macro_columns)

print("Final shape:", df_contract_final_no_spend.shape)
print("Rows missing macro match (using LPI_Score):", df_contract_final_no_spend["LPI_Score"].isna().sum())
print("Macro match rate:", f"{(1 - df_contract_final_no_spend['LPI_Score'].isna().mean()):.2%}")

display(df_contract_final_no_spend.head())

Added macro columns:
['company_country_clean', 'Country', 'Code', 'LPI_Score', 'Customs', 'Infrastructure', 'International_Shipments', 'Logistics_Competence', 'Tracking_Tracing', 'Timeliness', 'PPI_Value']
Final shape: (9201, 147)
Rows missing macro match (using LPI_Score): 67
Macro match rate: 99.27%


,contract_id,contract_number,contract_name,contract_status,contract_owner_cost_centre,terminated,term_type,start_date,expiration_date,supplier_id,...,Country,Code,LPI_Score,Customs,Infrastructure,International_Shipments,Logistics_Competence,Tracking_Tracing,Timeliness,PPI_Value
0,9675,9675,Bioreliance_Master_2018_MSA,published,7756.0,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,...,Denmark,DNK,3.991859,3.918058,3.95873,3.530159,4.007843,4.176078,4.407843,84.4
1,9675,9675,Bioreliance_Master_2018_MSA,published,7756.0,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,...,Denmark,NaN,3.991859,3.918058,3.95873,3.530159,4.007843,4.176078,4.407843,83.9
2,9675,9675,Bioreliance_Master_2018_MSA,published,7756.0,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,...,Denmark,NaN,3.991859,3.918058,3.95873,3.530159,4.007843,4.176078,4.407843,79.9
3,9675,9675,Bioreliance_Master_2018_MSA,published,7756.0,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,...,Denmark,NaN,3.991859,3.918058,3.95873,3.530159,4.007843,4.176078,4.407843,100.0
4,9675,9675,Bioreliance_Master_2018_MSA,published,7756.0,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,...,Denmark,NaN,3.991859,3.918058,3.95873,3.530159,4.007843,4.176078,4.407843,147.0


## Manual gold-label table

This is meant to be easy to maintain.

Add or remove contract ids directly inside the dictionary below.  
Each contract id should appear only once overall.

In [144]:
dept_counts = (
    df_contract_year
    .groupby("department")
    .agg(
        total_contracts=("contract_id", "nunique")
    )
    .sort_values("total_contracts", ascending=False)
)

display(dept_counts)

,total_contracts
department,
Devices & Needles,476
Drug Product Outsourcing,300
Drug Substance Outsourcing,298
Packaging Material,289
Raw Materials & Energy,281
"Quality, Production Services & Supplies",215
Bioprocessing & Raw Materials,113
Logistics,94
Bioprocessing and Excipients,59


## Expand manual Gold Labels: `Devices & Needles`

The goal of this step is to expand the manual gold-label set for Stage 2.

We focus on **Devices & Needles** because it has a large number of contracts and already contains initial expert-provided label examples. To avoid labeling repeated yearly observations of the same contract, we first reduce the contract-year table to **one latest row per contract**.

We then create two candidate pools:

- **Yes candidates**: contracts whose titles suggest contractual change, revision, replacement, pricing changes, outsourcing, or other renegotiation-like events.
- **No candidates**: contracts that look like stable, routine agreements with no obvious indication of contractual change.

These candidate pools are only used for manual inspection. Final labels are still assigned conservatively based on judgment.

In [145]:
# Step 1 — Focus on one department and keep one latest row per contract

dept_name = "Devices & Needles"

df_devices = df_contract_final_no_spend[
    df_contract_final_no_spend["department"] == dept_name
].copy()

df_devices_latest = (
    df_devices
    .sort_values(["contract_id", "observation_year"])
    .drop_duplicates(subset=["contract_id"], keep="last")
    .copy()
)

print("Department:", dept_name)
print("Original contract-year rows:", df_devices.shape[0])
print("Unique contracts:", df_devices["contract_id"].nunique())
print("Latest snapshot rows:", df_devices_latest.shape[0])

display(
    df_devices_latest[
        ["contract_id", "contract_number", "contract_name", "supplier_display_name", "observation_year"]
    ].head(10)
)

Department: Devices & Needles
Original contract-year rows: 2197
Unique contracts: 476
Latest snapshot rows: 476


,contract_id,contract_number,contract_name,supplier_display_name,observation_year
8914,100947,100956,"Baumann_Springs_Ltd_Call_off_2020_OA_AA4_APP1,13",BAUMANN SPRINGS LTD,2025
8166,101614,101625,NIPRO CORPORATION_CALL-OFF_2021_OA_AA_4,NIPRO CORPORATION,2025
8885,102698,102710,Nypro_Call-off_2020_EA_AA2_APP23,NYPRO INC DBA JABIL,2025
8679,105396,105408,Terumo Europe N.V._Stand-Alone_2015_BTA,TERUMO EUROPE N.V. RESEARCHPARK ZONE 2 HAASRODE,2025
8419,106130,106142,Mesa Parts GmbH_Call-off_20210202_SA_AA,MESA PARTS GMBH,2025
3899,109078,109090,MFS_Call-off_20210212_OA_APP,MFS Technology (S) Pte Ltd,2025
8261,111853,111865,WMT_Call-off_2021_PO7776000018_290M710_Housing...,MGS LYNGE A/S,2025
8236,111860,111872,WMT_Call-off_2021_PO7776000017_290M710-02_Hous...,MGS LYNGE A/S,2025
8241,111869,111881,WMT_Call-off_2021_PO7776000016_290M710-03_Hous...,MGS LYNGE A/S,2025
1961,1119,1119,BSN Medical AB_Master_20190213_SA,BSN MEDICAL AB,2025


## Define Inspection Columns

To support manual labeling, we inspect a compact subset of columns that are useful for judging whether a contract looks like a renegotiation case or a stable baseline agreement.

In [146]:
inspection_cols = [
    "contract_id",
    "contract_number",
    "contract_name",
    "supplier_display_name",
    "contract_status",
    "term_type",
    "contract_value_dkk",
    "payment_terms",
    "incoterms",
    "years_to_expiry",
    "contract_age_years",
    "fin_risk_level",
    "fin_financial_risk_score",
    "esg_risk_level",
    "esg_esg_overall",
    "news_article_count",
    "news_negative_ratio",
    "news_has_high_relevance_negative_news",
    "avg_market_cap",
    "market_beta_1y",
    "price_volatility_score",
    "PPI_Value",
]

inspection_cols = [c for c in inspection_cols if c in df_devices_latest.columns]

print("Inspection columns used:")
print(inspection_cols)

Inspection columns used:
['contract_id', 'contract_number', 'contract_name', 'supplier_display_name', 'contract_status', 'term_type', 'contract_value_dkk', 'payment_terms', 'incoterms', 'years_to_expiry', 'contract_age_years', 'fin_risk_level', 'fin_financial_risk_score', 'esg_risk_level', 'esg_esg_overall', 'news_article_count', 'news_negative_ratio', 'news_has_high_relevance_negative_news', 'avg_market_cap', 'market_beta_1y', 'price_volatility_score', 'PPI_Value']


## Build a **Yes** Candidate Pool

To find likely positive examples, we prioritize contracts whose titles explicitly suggest contractual change, such as:

- replacement
- restated agreement
- amendment
- pricing agreement
- outsourcing
- extension
- revision

We also remove obvious call-off style contracts and zero-value placeholders to improve label quality.

In [147]:
# Step 2 — Build YES candidates

yes_keywords = r"Replacement|Restated|Amendment|Pricing|Outsourcing|Extension|Revision"

df_yes_candidates = df_devices_latest[
    df_devices_latest["contract_name"].str.contains(
        yes_keywords,
        case=False,
        na=False
    )
].copy()

# Remove obvious operational / repetitive sub-contracts
df_yes_candidates = df_yes_candidates[
    ~df_yes_candidates["contract_name"].str.contains(
        "CALL|Call-off",
        case=False,
        na=False
    )
].copy()

# Optional quality filter: remove zero-value contracts if present
if "contract_value_dkk" in df_yes_candidates.columns:
    df_yes_candidates = df_yes_candidates[
        df_yes_candidates["contract_value_dkk"].fillna(0) > 0
    ].copy()

# Sort for easier review
sort_cols_yes = [c for c in ["contract_value_dkk", "fin_financial_risk_score", "news_negative_ratio"] if c in df_yes_candidates.columns]
if sort_cols_yes:
    df_yes_candidates = df_yes_candidates.sort_values(by=sort_cols_yes, ascending=False)

print("YES candidate contracts:", df_yes_candidates.shape[0])
display(df_yes_candidates[inspection_cols].head(20))

YES candidate contracts: 17


,contract_id,contract_number,contract_name,supplier_display_name,contract_status,term_type,contract_value_dkk,payment_terms,incoterms,years_to_expiry,...,fin_financial_risk_score,esg_risk_level,esg_esg_overall,news_article_count,news_negative_ratio,news_has_high_relevance_negative_news,avg_market_cap,market_beta_1y,price_volatility_score,PPI_Value
2992,352237,353253,Ypsomed_SA_2023_Replacement of existing SA,YPSOMED AG,published,perpetual,3.500000e+09,F030 Invoice Date + 30 days,DDP Delivered duty paid,74,...,NaN,NaN,NaN,13.0,0.307692,1.0,NaN,NaN,NaN,148.3
2941,438366,439086,FLEX LTD_OSA_2024_Outsourcing Services Agreement,FLEXTRONICS INTERNATIONAL USA INC,published,perpetual,2.318000e+09,F045 Invoice Date + 45 days,FCA Free carrier,74,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,148.3
3000,301059,302323,SHL_Master_2023__Restated SA,SHL MEDICAL AG,published,perpetual,1.773000e+09,F060 Invoice Date + 60 days,FCA Free carrier,74,...,NaN,NaN,NaN,14.0,0.000000,0.0,NaN,NaN,NaN,148.3
3806,518137,517783,Steiert Pricing Agreement Appendix 2025 Inject...,STEIERT PRÄZISIONSFORMENBAU GMBH,published,perpetual,3.129000e+08,F060 Invoice Date + 60 days,DAP Del.at Place-excl.unloading,74,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,148.3
551,587975,586829,Amendment 1 Price Agreement - Ace Mold - Frame...,"ACE MOLD INDUSTRIAL(SHENZHEN) CO.,L",published,perpetual,1.005000e+08,F060 Invoice Date + 60 days,DAP Del.at Place-excl.unloading,74,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,148.3
2826,372096,372991,Ypsomed_SA_2023_Amendment1,YPSOMED AG,published,perpetual,7.450000e+07,F060 Invoice Date + 60 days,DDP Delivered duty paid,74,...,NaN,NaN,NaN,13.0,0.307692,1.0,NaN,NaN,NaN,148.3
603,594944,593686,Noble International_Master_20190813_MSA_Amendm...,"NOBLE INTERNATIONAL, INC.",published,perpetual,4.410000e+07,F045 Invoice Date + 45 days,NaN,74,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,148.3
584,530843,530338,Flextronics Medical Sales and Marketing Ltd._2...,FLEXTRONICS MARKETING (L) LTD,published,perpetual,3.196648e+07,F045 Invoice Date + 45 days,DDP Delivered duty paid,74,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,148.3
4528,373252,374135,Gerresheimer_2023_Amendment03_FxT Project Agre...,GERRESHEIMER REGENSBURG GMBH,published,perpetual,1.740320e+07,F060 Invoice Date + 60 days,EXW Ex works,74,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,148.3
4459,285657,287176,Gerresheimer_2022_Amendment01_FxT Project Agre...,GERRESHEIMER REGENSBURG GMBH,published,perpetual,1.341000e+07,F060 Invoice Date + 60 days,DAP Del.at Place-excl.unloading,74,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,148.3


## Build a **No** Candidate Pool

To find likely negative examples, we look for contracts that do **not** show obvious signs of renegotiation or contractual revision.

These are intended to represent stable, routine agreements rather than revised or renegotiated contracts.

In [148]:
# Step 3 — Build NO candidates

change_keywords = r"Replacement|Restated|Amendment|Pricing|Outsourcing|Extension|Revision"

df_no_candidates = df_devices_latest[
    ~df_devices_latest["contract_name"].str.contains(
        change_keywords,
        case=False,
        na=False
    )
].copy()

# Remove obvious call-off style contracts
df_no_candidates = df_no_candidates[
    ~df_no_candidates["contract_name"].str.contains(
        "CALL|Call-off",
        case=False,
        na=False
    )
].copy()

# Optional quality filter: remove zero-value contracts if present
if "contract_value_dkk" in df_no_candidates.columns:
    df_no_candidates = df_no_candidates[
        df_no_candidates["contract_value_dkk"].fillna(0) > 0
    ].copy()

# Sort to surface stable-looking contracts
sort_cols_no = [c for c in ["fin_financial_risk_score", "contract_age_years", "contract_value_dkk"] if c in df_no_candidates.columns]
ascending_no = [True, False, True][:len(sort_cols_no)] if sort_cols_no else None

if sort_cols_no:
    # make ascending list match actual sort columns
    ascending_map = {
        "fin_financial_risk_score": True,
        "contract_age_years": False,
        "contract_value_dkk": True,
    }
    ascending_no = [ascending_map[c] for c in sort_cols_no]
    df_no_candidates = df_no_candidates.sort_values(by=sort_cols_no, ascending=ascending_no)

print("NO candidate contracts:", df_no_candidates.shape[0])
display(df_no_candidates[inspection_cols].head(20))

NO candidate contracts: 124


,contract_id,contract_number,contract_name,supplier_display_name,contract_status,term_type,contract_value_dkk,payment_terms,incoterms,years_to_expiry,...,fin_financial_risk_score,esg_risk_level,esg_esg_overall,news_article_count,news_negative_ratio,news_has_high_relevance_negative_news,avg_market_cap,market_beta_1y,price_volatility_score,PPI_Value
3241,177,177,SABIC Innovative Plastics B.V.P_Master_2015110...,SABIC INNOVATIVE PLASTICS B.V.,published,perpetual,1.847600e+01,"O045 14 days -1,5%, 45 days",CIP Carriage and insurance paid,74,...,NaN,NaN,NaN,1.0,1.0,1.0,NaN,NaN,NaN,148.3
3389,1614,1614,Nipro Corporation_Master_20160226_OA,NIPRO CORPORATION,published,perpetual,1.070000e+09,F060 Invoice Date + 60 days,FOB Free on board,74,...,NaN,NaN,NaN,6.0,0.0,0.0,NaN,NaN,NaN,148.3
3295,1546,1546,"Value Sun Trading Co., Ltd._Stand-Alone_2017_BTA",沈阳立伟信科技有限公司,published,perpetual,6.200000e+04,F030 Invoice Date + 30 days,CFR Costs and freight,74,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,148.3
3198,1508,1508,OSM Holding AB_Master_20170227_SA,东莞尚科思工艺制品有限公司,published,perpetual,7.800000e+06,L045 Current Month + 45 days,DAP Del.at Place-excl.unloading,74,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,148.3
3410,1605,1605,Terumo Europe N.V._Master_20170401_SA,TERUMO EUROPE N.V. RESEARCHPARK ZONE 2 HAASRODE,published,perpetual,2.980000e+07,F030 Invoice Date + 30 days,DAP Del.at Place-excl.unloading,74,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,148.3
485,4507,4507,Nolato_Master_20170601_OA,NOLATO MEDITECH AB,published,perpetual,7.333980e+07,F060 Invoice Date + 60 days,DAP Del.at Place-excl.unloading,74,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,148.3
3315,4816,4817,SHL_Master_20170821_SA,SHL MEDICAL AG,published,perpetual,7.800000e+08,NaN,FCA Free carrier,74,...,NaN,NaN,NaN,14.0,0.0,0.0,NaN,NaN,NaN,148.3
3286,1545,1545,"Kyokuto Trading (Shanghai) Co., Ltd_STAND-ALON...",极东贸易（上海）有限公司,published,perpetual,1.953000e+06,F045 Invoice Date + 45 days,DDP Delivered duty paid,74,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,148.3
3278,1603,1603,Nemera La Verpilliere_Master_20181219_OA,NEMERA LA VERPILLIÉRE,published,perpetual,1.937000e+08,F060 Invoice Date + 60 days,DAP Del.at Place-excl.unloading,74,...,NaN,Go ahead,3.0,NaN,NaN,NaN,NaN,NaN,NaN,148.3
3547,4846,4847,Noble_CDA_20190401_Three way agreement with SH...,"NOBLE INTERNATIONAL, INC.",published,NaN,6.300000e+00,NaN,NaN,74,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,148.3


### Rationale for Selected Additional Gold Labels: `Devices & Needles`

To expand the manual gold-label set for the **Devices & Needles** department, contracts were reviewed using a conservative naming-based labeling strategy. Since the available numerical risk indicators were not strongly discriminative within this subset, `contract titles were used as the primary signal` for identifying whether a contract more likely represented a **renegotiation / revision case** (**Yes**) or a **stable baseline agreement** (**No**).

#### Selected **`Yes`** contracts

The following contracts were selected as additional **Yes** examples because their titles explicitly suggest contractual change, revision, or commercial adjustment:

- **353253 — Ypsomed_SA_2023_Replacement of existing SA**  
  Chosen because the phrase *“replacement of existing”* strongly indicates that a prior agreement was substituted or revised, which is closely aligned with the thesis definition of renegotiation.

- **302323 — SHL_Master_2023__Restated SA**  
  Chosen because *“restated”* suggests that the agreement was formally revised and reissued, indicating updated contractual terms or structure.

- **517783 — Steiert Pricing Agreement Appendix 2025 ...**  
  Chosen because *“pricing agreement”* explicitly points to commercial term revision, which is one of the clearest indicators of renegotiation-related activity.

#### Optional additional **Yes** contracts

These contracts were also identified as plausible positive examples because they contain explicit wording associated with contractual revision:

- **372991 — Ypsomed_SA_2023_Amendment1**  
  The term *“amendment”* directly indicates a formal modification to an existing agreement.

- **586829 — Amendment 1 Price Agreement - Ace Mold ...**  
  Contains both *“amendment”* and *“price agreement”*, making it a strong candidate for a renegotiation-type case.


#### Selected **`No`** contracts

The following contracts were selected as additional **No** examples because their titles resemble stable, routine baseline agreements and do not contain explicit signals of revision, amendment, replacement, or pricing change:

- **1614 — Nipro Corporation_Master_20160226_OA**  
  Chosen as a standard master agreement with no visible indication of contractual change.

- **1603 — Nemera La Verpilliere_Master_20181219_OA**  
  Chosen as a routine master agreement without wording suggesting renegotiation or restructuring.

- **4507 — Nolato_Master_20170601_OA**  
  Chosen as a stable-looking long-running master agreement with no explicit revision signal.

#### Optional additional **No** contracts

These were also considered suitable baseline non-renegotiation examples:

- **1605 — Terumo Europe N.V._Master_20170401_SA**  
  Appears to be a standard master agreement without obvious evidence of modification.

- **1508 — OSM Holding AB_Master_20170227_SA**  
  Also resembles a routine long-standing agreement without clear renegotiation indicators.



## `Packaging Material`

In [149]:
# Step 1 — Filter Packaging Material and keep one latest row per contract

dept_name = "Packaging Material"

df_packaging = df_contract_final_no_spend[
    df_contract_final_no_spend["department"] == dept_name
].copy()

df_packaging_latest = (
    df_packaging
    .sort_values(["contract_id", "observation_year"])
    .drop_duplicates(subset=["contract_id"], keep="last")
    .copy()
)

print("Department:", dept_name)
print("Original rows:", df_packaging.shape[0])
print("Unique contracts:", df_packaging["contract_id"].nunique())
print("Latest snapshot rows:", df_packaging_latest.shape[0])

display(
    df_packaging_latest[
        ["contract_id", "contract_number", "contract_name", "supplier_display_name", "observation_year"]
    ].head(10)
)

Department: Packaging Material
Original rows: 1272
Unique contracts: 289
Latest snapshot rows: 289


,contract_id,contract_number,contract_name,supplier_display_name,observation_year
8895,100101,100110,Gerresheimer Vaerloese_Call-off_2021_AA1_APP_B...,GERRESHEIMER VAERLOESE A/S,2025
7738,10528,10529,West Pharmaceutical Services_Stand-alone_2019-...,WEST PHARMACEUTICAL SERVICES DEUTSCHLAND GMBH ...,2025
7752,10540,10541,West Pharmaceutical Services_Stand-alone_2019-...,WEST PHARMACEUTICAL SERVICES DEUTSCHLAND GMBH ...,2025
7724,10545,10546,West Pharmaceutical Services_Stand-alone_2019-...,WEST PHARMACEUTICAL SERVICES DEUTSCHLAND GMBH ...,2025
7686,10553,10554,West Pharmaceutical Services_Stand-alone_2019-...,WEST PHARMACEUTICAL SERVICES DEUTSCHLAND GMBH ...,2025
7717,10556,10557,West Pharmaceutical Services_Stand-alone_2019_...,WEST PHARMACEUTICAL SERVICES DEUTSCHLAND GMBH ...,2025
9154,114696,114708,Gerresheimer Vaerloese_Call-off_2021_US-to-US ...,GERRESHEIMER VAERLOESE A/S,2025
328,118119,118130,"West Pharmaceutical Services, Inc._Call-off_20...",WEST PHARMACEUTICAL SERVICES INC,2025
8819,120163,120174,Schur Pack Denmark_Stand-alone_2021_BTA,SCHUR PACK DENMARK A/S,2025
2515,1245,1245,"New Century Graphic Arts Industrial Co., Ltd._...",大连新世纪印刷信息产业有限公司,2025


In [150]:
yes_keywords = r"Replacement|Restated|Amendment|Pricing|Price|Extension|Revision|Adjustment"

df_yes_candidates = df_packaging_latest[
    df_packaging_latest["contract_name"].str.contains(
        yes_keywords,
        case=False,
        na=False
    )
].copy()

# remove call-offs
df_yes_candidates = df_yes_candidates[
    ~df_yes_candidates["contract_name"].str.contains(
        "CALL|Call-off",
        case=False,
        na=False
    )
]

df_yes_candidates = df_yes_candidates.sort_values(
    by="contract_value_dkk",
    ascending=False
)

print("YES candidate contracts:", df_yes_candidates.shape[0])

display(
    df_yes_candidates[
        [
            "contract_id",
            "contract_number",
            "contract_name",
            "supplier_display_name",
            "contract_value_dkk"
        ]
    ].head(20)
)

YES candidate contracts: 52


,contract_id,contract_number,contract_name,supplier_display_name,contract_value_dkk
4836,299901,301141,Reintegrate_European Energy_Restated Supply Ag...,RE::INTEGRATE APS,395000000.0
2709,260438,261718,Gerresheimer_Amendment 5_2022_Capacity 2023-2025,GERRESHEIMER BÜNDE GMBH,342700000.0
2775,493096,492795,MM Group_Amendment 2_2025_Terms update and 202...,MM ESON PAC AB,120000000.0
553,510361,510287,CCL Label_Amendment_2025_Label Tender Outcome ...,CCL LABEL A/S,42000000.0
393,515335,515133,Autajon_Amendment 5_2025_Label Tender,AUTAJON SP,35760000.0
578,493105,492804,MM Group_Amendment 1_2025_Update to supplement...,MM ESON PAC AB,8000000.0
576,499275,499086,Dobber Healthcare_Amendment_2025_Upate of term...,DOBBER HEALTHCARE LEAFLETS,3432823.0
398,515322,515117,MM Group_Amendment_2025:Label Tender Allocation,MM ESON PAC AB,3400000.0
392,536317,535752,All4Labels_Amendment_2025_Label Tender Outcome...,ALL4LABELS DENMARK SB A/S,1000000.0
5999,540852,540232,Goncalves Industria Grafica_ Amendment no.3_2025,GONCALVES S A INDUSTRIA GRAFICA,0.0


In [151]:
change_keywords = r"Replacement|Restated|Amendment|Pricing|Price|Extension|Revision|Adjustment"

df_no_candidates = df_packaging_latest[
    ~df_packaging_latest["contract_name"].str.contains(
        change_keywords,
        case=False,
        na=False
    )
].copy()

# remove call-offs
df_no_candidates = df_no_candidates[
    ~df_no_candidates["contract_name"].str.contains(
        "CALL|Call-off",
        case=False,
        na=False
    )
]

df_no_candidates = df_no_candidates.sort_values(
    by="contract_value_dkk",
    ascending=False
)

print("NO candidate contracts:", df_no_candidates.shape[0])

display(
    df_no_candidates[
        [
            "contract_id",
            "contract_number",
            "contract_name",
            "supplier_display_name",
            "contract_value_dkk"
        ]
    ].head(20)
)

NO candidate contracts: 140


,contract_id,contract_number,contract_name,supplier_display_name,contract_value_dkk
2916,428822,429598,MM Group_Master_2024_SA_Denmark_US_France,MM ESON PAC AB,6.530000e+08
3361,1984,1984,"WestRock MWV, LLC_Master_20160930_SA",SMURFIT WESTROCK,1.117500e+08
3368,190,190,Tonnellier_Master_ 20150204_SA,TONNELLIER IMPRIMEUR S.A.S,4.470000e+07
4738,212122,212222,Körber Pharma Packaging Materials_Master_2022_SA,KÖRBER PHARMA PACKAGING AG,9.685000e+06
3968,195289,195301,DS Smith_Master_2022_SA_Corrugated Boxes,NaN,9.563402e+06
543,536217,535659,Smurfit Westrock Brazil_Master_2025_SA_Shippin...,SMURFIT KAPPA DO BRASIL INDUSTRIA DE EMBALAGEN...,8.350000e+06
4033,178559,178575,United Creation Packaging Solution (Tianjin)_M...,合众创亚（天津）包装有限公司,6.476119e+06
4490,370825,371766,Multi Packaging Solutions_AA3_2023_Appendices ...,"MULTI PACKAGING SOLUTIONS, INC.",6.300000e+06
4464,364088,365065,New Print Package and Displays_AA3_2023_Append...,NEW PRINT EMBALAGENS E DISPLAYS LTD,6.000000e+06
4002,156818,156827,Carolina Container_Master_2021_SA,Carolina Container Co.,5.725642e+06


### Manual Gold Label Selection — Packaging Material

### Selected YES Contracts (Renegotiation Events)

The following contracts were labeled as **YES**, as they explicitly contain signals of renegotiation such as amendments, term updates, or tender outcomes.

**Contract 260438 — Gerresheimer Amendment (Capacity Planning)**  
- Contract name: *Gerresheimer_Amendment 5_2022_Capacity 2023–2025*  
- Supplier: GERRESHEIMER BÜNDE GMBH  
- Rationale:  
  - Explicit amendment to an existing agreement  
  - Capacity planning adjustment over multiple years  
  - Represents a strategic operational change in supply terms  

**Contract 493096 — MM Group Terms Update**  
- Contract name: *MM Group_Amendment 2_2025_Terms update*  
- Supplier: MM ESON PAC AB  
- Rationale:  
  - Direct modification of contractual terms  
  - Indicates renegotiation of existing conditions rather than creation of a new contract  

**Contract 515335 — Autajon Label Tender Amendment**  
- Contract name: *Autajon_Amendment 5_2025_Label Tender*  
- Supplier: AUTAJON SP  
- Rationale:  
  - Linked to a tender outcome  
  - Suggests renegotiation of pricing, allocation, or supply conditions  
  - Typical example of supplier renegotiation following competitive sourcing  

These contracts represent clear and interpretable renegotiation scenarios and therefore serve as reliable positive examples for model training.


### Selected NO Contracts (Baseline Agreements)

The following contracts were labeled as **NO**, as they represent standard supplier agreements without evidence of renegotiation.

**Contract 428822 — MM Group Master Agreement**  
- Contract name: *MM Group_Master_2024_SA_Denmark_US_France*  
- Supplier: MM ESON PAC AB  
- Rationale:  
  - Standard master agreement  
  - No amendment or modification signal  
  - Represents a baseline supplier relationship  

**Contract 1984 — WestRock Master Agreement**  
- Contract name: *WestRock MWV, LLC_Master_20160930_SA*  
- Supplier: SMURFIT WESTROCK  
- Rationale:  
  - Stable long-term supply agreement  
  - No indication of renegotiation or contract revision  

**Contract 212122 — Körber Pharma Packaging Materials Agreement**  
- Contract name: *Körber Pharma Packaging Materials_Master_2022_SA*  
- Supplier: KÖRBER PHARMA PACKAGING AG  
- Rationale:  
  - Standard supply contract  
  - Serves as a representative example of a non-renegotiated agreement  



## Expand `Logistics` labels

In [152]:
dept_name = "Logistics"

# 1. Filter Logistics from the final joined table
df_logistics = df_contract_final_no_spend[
    df_contract_final_no_spend["department"] == dept_name
].copy()

print("Original rows:", df_logistics.shape[0])
print("Unique contracts:", df_logistics["contract_id"].nunique())

# 2. Keep one latest row per contract
df_logistics_latest = (
    df_logistics
    .sort_values(["contract_id", "observation_year"])
    .drop_duplicates(subset=["contract_id"], keep="last")
    .copy()
)

print("Latest snapshot rows:", df_logistics_latest.shape[0])

# 3. Inspect important columns
display(
    df_logistics_latest[
        [
            "contract_id",
            "contract_name",
            "supplier_display_name",
            "contract_value_dkk",
            "payment_terms",
            "incoterms",
            "term_type",
            "years_to_expiry",
            "contract_age_years",
            "expiry_pressure_bucket",
            "fin_financial_risk_score",
            "esg_risk_level",
            "news_negative_ratio",
        ]
    ]
    .sort_values("contract_value_dkk", ascending=False)
    .head(30)
)

# 4. Helpful summaries
display(df_logistics_latest["expiry_pressure_bucket"].value_counts())
display(df_logistics_latest["years_to_expiry"].describe())
display(df_logistics_latest["payment_terms"].value_counts().head(10))

Original rows: 282
Unique contracts: 94
Latest snapshot rows: 94


,contract_id,contract_name,supplier_display_name,contract_value_dkk,payment_terms,incoterms,term_type,years_to_expiry,contract_age_years,expiry_pressure_bucket,fin_financial_risk_score,esg_risk_level,news_negative_ratio
394,505649,DSV Air and Sea Holding_MSA_2025_Clinical Logi...,DSV AIR & SEA HOLDING A/S,2.680000e+08,F060 Invoice Date + 60 days,NaN,fixed,3,0,1_to_3y,NaN,NaN,0.000000
359,322164,Troy X_NNAS_Master_2023_Commercial Lease Agree...,Purchasing - Dummy Supplier,1.836000e+08,F060 Invoice Date + 60 days,NaN,fixed,8,2,5y_plus,NaN,NaN,NaN
5094,515275,DSV_LSPA_2025_PS PAC 3 Warehouse,DSV SOLUTIONS A/S,1.830000e+08,F060 Invoice Date + 60 days,NaN,fixed,5,0,3_to_5y,NaN,Do not source,0.266667
346,340383,DSV_Master_2023_LSPA_warehouse service,DSV SOLUTIONS A/S,1.700000e+08,F060 Invoice Date + 60 days,NaN,fixed,3,2,1_to_3y,NaN,Do not source,0.266667
397,587361,Kuehne + Nagel_Amendment 8_2025_4PL MSA extens...,KUEHNE + NAGEL S.À R.L,1.117500e+08,F045 Invoice Date + 45 days,NaN,fixed,5,0,3_to_5y,NaN,NaN,NaN
429,192312,SkyNRG_Stand-Alone_2021_Partnership Agreement ...,SKYNRG,1.117500e+08,F060 Invoice Date + 60 days,DDP Delivered duty paid,auto_renew,4,4,3_to_5y,NaN,Do not source,0.000000
450,332780,MDLogistics_Call-off_2023_NNAS_Primary_US Road...,MDLOGISTICS,6.100000e+07,F045 Invoice Date + 45 days,EXW Ex works,fixed,2,2,1_to_3y,NaN,NaN,NaN
5089,470427,DHL_Call-Off no. 7_PS Pac 1/2 Extension_2024,DHL EXEL SUPPLY CHAIN (DENMARK) A/S,5.375000e+07,F060 Invoice Date + 60 days,EXW Ex works,fixed,1,1,within_1y,NaN,NaN,NaN
362,335471,DHL Global Forwarding_Call-off_2023_NNAS_Prima...,DHL GLOBAL FORWARDING (BRAZIL) LOGISTICS LTDA,5.200000e+07,F060 Invoice Date + 60 days,DDP Delivered duty paid,fixed,1,2,within_1y,NaN,NaN,NaN
5153,582913,MDLogistics_Amendment 3_2025_Extension of the ...,MDLOGISTICS,4.690000e+07,F060 Invoice Date + 60 days,NaN,fixed,2,0,1_to_3y,NaN,NaN,NaN


expiry_pressure_bucket
5y_plus      41
1_to_3y      20
within_1y    17
3_to_5y      16
Name: count, dtype: int64

count    94.000000
mean     32.351064
std      35.287363
min       1.000000
25%       2.000000
50%       5.000000
75%      74.000000
max      74.000000
Name: years_to_expiry, dtype: float64

payment_terms
F060 Invoice Date + 60 days    43
F030 Invoice Date + 30 days    22
F045 Invoice Date + 45 days    10
F015 Invoice Date + 15 days     2
F014 Invoice Date + 14 days     1
Name: count, dtype: int64

In [153]:
df_logistics["contract_id_clean"] = (
    df_logistics["contract_id"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.strip()
)

proposed_logistics_yes = [
    "587361",
    "470427",
    "456272",
    "550966",
    "508089",
]

proposed_logistics_no = [
    "505649",
    "515275",
    "340383",
    "432542",
    "575807",
]


display(
    df_logistics[
        df_logistics["contract_id_clean"].isin(proposed_logistics_yes)
    ][
        [
            "contract_id",
            "contract_name",
            "supplier_display_name",
            "contract_value_dkk",
            "payment_terms",
            "incoterms",
            "term_type",
            "years_to_expiry",
            "contract_age_years",
            "expiry_pressure_bucket",
            "esg_risk_level",
            "news_negative_ratio",
        ]
    ].sort_values("contract_value_dkk", ascending=False)
)


display(
    df_logistics[
        df_logistics["contract_id_clean"].isin(proposed_logistics_no)
    ][
        [
            "contract_id",
            "contract_name",
            "supplier_display_name",
            "contract_value_dkk",
            "payment_terms",
            "incoterms",
            "term_type",
            "years_to_expiry",
            "contract_age_years",
            "expiry_pressure_bucket",
            "esg_risk_level",
            "news_negative_ratio",
        ]
    ].sort_values("contract_value_dkk", ascending=False)
)



,contract_id,contract_name,supplier_display_name,contract_value_dkk,payment_terms,incoterms,term_type,years_to_expiry,contract_age_years,expiry_pressure_bucket,esg_risk_level,news_negative_ratio
397,587361,Kuehne + Nagel_Amendment 8_2025_4PL MSA extens...,KUEHNE + NAGEL S.À R.L,111750000.0,F045 Invoice Date + 45 days,NaN,fixed,5,0,3_to_5y,NaN,NaN
5088,470427,DHL_Call-Off no. 7_PS Pac 1/2 Extension_2024,DHL EXEL SUPPLY CHAIN (DENMARK) A/S,53750000.0,F060 Invoice Date + 60 days,EXW Ex works,fixed,2,0,1_to_3y,NaN,NaN
5089,470427,DHL_Call-Off no. 7_PS Pac 1/2 Extension_2024,DHL EXEL SUPPLY CHAIN (DENMARK) A/S,53750000.0,F060 Invoice Date + 60 days,EXW Ex works,fixed,1,1,within_1y,NaN,NaN
5081,456272,DSV_Call-off no. 5_2024_LSPA_Capacity Expansion,DSV SOLUTIONS A/S,12000000.0,F060 Invoice Date + 60 days,EXW Ex works,perpetual,75,0,5y_plus,Do not source,0.000000
5082,456272,DSV_Call-off no. 5_2024_LSPA_Capacity Expansion,DSV SOLUTIONS A/S,12000000.0,F060 Invoice Date + 60 days,EXW Ex works,perpetual,74,1,5y_plus,Do not source,0.266667
5103,550966,Geodis DSD_Call-off_no.2 Extension Extra Capac...,GEODIS CL ILE DE FRANCE,10520000.0,F060 Invoice Date + 60 days,DDP Delivered duty paid,fixed,2,0,1_to_3y,Go ahead,NaN
5104,508089,DSV_Call-Off_2025_Amendment03_Additional services,DSV SOLUTIONS A/S,2600000.0,F060 Invoice Date + 60 days,NaN,fixed,3,0,1_to_3y,Do not source,0.266667


,contract_id,contract_name,supplier_display_name,contract_value_dkk,payment_terms,incoterms,term_type,years_to_expiry,contract_age_years,expiry_pressure_bucket,esg_risk_level,news_negative_ratio
394,505649,DSV Air and Sea Holding_MSA_2025_Clinical Logi...,DSV AIR & SEA HOLDING A/S,268000000.0,F060 Invoice Date + 60 days,NaN,fixed,3,0,1_to_3y,NaN,0.000000
5094,515275,DSV_LSPA_2025_PS PAC 3 Warehouse,DSV SOLUTIONS A/S,183000000.0,F060 Invoice Date + 60 days,NaN,fixed,5,0,3_to_5y,Do not source,0.266667
344,340383,DSV_Master_2023_LSPA_warehouse service,DSV SOLUTIONS A/S,170000000.0,F060 Invoice Date + 60 days,NaN,fixed,5,0,3_to_5y,NaN,0.000000
345,340383,DSV_Master_2023_LSPA_warehouse service,DSV SOLUTIONS A/S,170000000.0,F060 Invoice Date + 60 days,NaN,fixed,4,1,3_to_5y,Do not source,0.000000
346,340383,DSV_Master_2023_LSPA_warehouse service,DSV SOLUTIONS A/S,170000000.0,F060 Invoice Date + 60 days,NaN,fixed,3,2,1_to_3y,Do not source,0.266667
364,432542,Legendre_Master_NNAS_Primary Distribution_LSPA...,LEGENDRE SAS,45966500.0,F030 Invoice Date + 30 days,DDP Delivered duty paid,fixed,4,0,3_to_5y,NaN,NaN
365,432542,Legendre_Master_NNAS_Primary Distribution_LSPA...,LEGENDRE SAS,45966500.0,F030 Invoice Date + 30 days,DDP Delivered duty paid,fixed,3,1,1_to_3y,NaN,NaN
415,575807,World Courier_NN A/S_Clinical Logistics Servic...,WORLD COURIER DENMARK A/S,35000000.0,F060 Invoice Date + 60 days,CPT Carriage paid,fixed,3,0,1_to_3y,NaN,NaN


### Proposed Additional `Logistics` Labels

The following contracts were manually reviewed and assigned preliminary labels based on observable contract characteristics and domain guidance provided by the category manager. Each decision was made by inspecting the contract title and contextual variables in the dataset.

### Proposed **`Yes`** Contracts (Renegotiated)

These contracts were labeled as **Yes** because their titles explicitly indicate contractual modification or operational change consistent with renegotiation triggers described by the category manager.

**587361 — Kuehne + Nagel Amendment 8 2025 4PL MSA Extension**

the title clearly indicates both an amendment and an extension of an existing logistics agreement. Amendments and extensions represent formal modifications to contractual terms and are typical mechanisms through which renegotiation occurs.

- **470427 — DHL Call-Off no. 7 PS Pac 1/2 Extension 2024**

it explicitly references an extension. Contract extensions commonly occur during renewal cycles and provide an opportunity to revise pricing, capacity commitments, or service conditions. The contract also falls within the `within_1y` expiry category, which increases the likelihood of renegotiation activity.

- **456272 — DSV Call-off no. 5 2024 LSPA Capacity Expansion**

the title includes "Capacity Expansion," indicating an increase in operational requirements. The category manager explicitly identified increased capacity needs as a common trigger for renegotiation in logistics contracts.

- **550966 — Geodis DSD Call-off no. 2 Extension Extra Capacity**

it contains both an extension and additional capacity commitments. These signals suggest a modification of service scope and operational requirements, which aligns with the domain explanation that renegotiations frequently occur when service demand changes.

- **508089 — DSV Call-Off 2025 Amendment 03 Additional Services**

the title includes both "Amendment" and "Additional services." The addition of new services represents an expansion of contract scope, which typically requires renegotiation of terms, pricing, or performance conditions.


### Proposed **`No`** Contracts (Non-Renegotiated)

These contracts were labeled as **No** because they appear to represent stable baseline agreements without explicit evidence of renegotiation or contractual modification.

- **505649 — DSV Air and Sea Holding MSA 2025 Clinical Logistics**

it appears to be a standard master service agreement establishing baseline logistics services. The title does not indicate any amendment, extension, or operational change that would suggest renegotiation.

- **515275 — DSV LSPA 2025 PS PAC 3 Warehouse**

it appears to represent a standard warehouse service agreement. There is no explicit reference to contractual revision, additional services, or scope modification.

- **340383 — DSV Master 2023 LSPA Warehouse Service**

it represents a baseline warehouse service agreement. The title indicates a master contract rather than a modification to an existing agreement.

- **432542 — Legendre Master NNAS Primary Distribution**

it appears to be a standard distribution agreement without explicit signals of renegotiation. The contract structure suggests ongoing operational service rather than contractual change.

- **575807 — World Courier Clinical Logistics Services**

it represents a routine logistics services agreement. The title indicates normal operational service delivery without evidence of amendment, extension, or scope modification.


### Summary

The labeling decisions were made through manual inspection of contract titles and contextual variables in the dataset. Contracts were labeled as **Yes** when they showed explicit evidence of contractual modification, and as **No** when they appeared to represent stable baseline agreements without visible changes.

This contract-level documentation ensures transparency and reproducibility of the manual labeling process used to construct the gold-label dataset.

## Manual gold labels

The gold labels below are maintained manually as a small benchmark set.

Important:
- labels are matched on `contract_number`
- every label must survive all upstream filtering steps to appear in the final dataset
- a label that does not appear in `df_contract_final_no_spend` will not be usable in Stage 1 or Stage 2

In [154]:
gold_contract_map = {
    "Bioprocessing & Excipients - Cell Culture Media": {
        "yes": [504149, 612555],
        "no": [346312, 1877],
    },
    "Logistics": {
        "yes": [192325, 541269, 587361, 470427, 456272, 550966, 508089],
        "no": [536319, 598142, 505649, 515275, 340383, 432542, 575807],
    },
    "Global Strategic Outsourcing & Devices I Devices & Needles": {
        "yes": [4822, 557, 565, 301059, 352237, 518137],
        "no": [483178, 25777, 1614, 1603, 4507],
    },
    "Packaging Material": {
        "yes": [260438, 493096, 515335],
        "no": [428822, 1984, 212122],
    },
}

df_gold_manual = build_manual_gold_table(gold_contract_map)

print("Manual gold table shape:", df_gold_manual.shape)
display(df_gold_manual.sort_values(["gold_department", "gold_y", "contract_number"]))

Manual gold table shape: (35, 3)


,contract_number,gold_y,gold_department
3,1877,0,Bioprocessing & Excipients - Cell Culture Media
2,346312,0,Bioprocessing & Excipients - Cell Culture Media
0,504149,1,Bioprocessing & Excipients - Cell Culture Media
1,612555,1,Bioprocessing & Excipients - Cell Culture Media
27,1603,0,Global Strategic Outsourcing & Devices I Devic...
26,1614,0,Global Strategic Outsourcing & Devices I Devic...
25,25777,0,Global Strategic Outsourcing & Devices I Devic...
28,4507,0,Global Strategic Outsourcing & Devices I Devic...
24,483178,0,Global Strategic Outsourcing & Devices I Devic...
21,301059,1,Global Strategic Outsourcing & Devices I Devic...


## Validate gold labels against the final contract table

This check is critical. A manually labeled contract is only usable if its `contract_number`
still exists in the final filtered contract table.

The summary below tells us:
- which labels survive into the final dataset
- which labels are missing after upstream filtering
- whether we have enough positive and negative examples to make a balanced train/test split

In [155]:
df_gold_matches, df_gold_valid, df_gold_missing = validate_manual_gold_labels(
    df_gold_manual=df_gold_manual,
    df_contracts=df_contract_final_no_spend,
)

print("Manual labels provided:", len(df_gold_manual))
print("Valid labels found in final dataset:", len(df_gold_valid))
print("Missing labels after upstream filtering:", len(df_gold_missing))
print()

print("Valid label counts:")
display(
    df_gold_valid["gold_y"]
    .value_counts(dropna=False)
    .rename_axis("gold_y")
    .reset_index(name="count")
    .sort_values("gold_y", ascending=False)
)

print("Valid labels by department:")
display(
    df_gold_valid.groupby(["gold_department", "gold_y"])
    .size()
    .reset_index(name="count")
    .sort_values(["gold_department", "gold_y"], ascending=[True, False])
)

print("Matched gold contracts:")
display(
    df_gold_valid[
        ["contract_id", "contract_number", "contract_name", "department", "gold_y", "gold_department"]
    ]
    .drop_duplicates()
    .sort_values(["gold_department", "gold_y", "contract_number"], ascending=[True, False, True])
)

if len(df_gold_missing) > 0:
    print("Missing gold contracts:")
    display(
        df_gold_missing[
            ["contract_number", "gold_y", "gold_department"]
        ]
        .drop_duplicates()
        .sort_values(["gold_department", "gold_y", "contract_number"], ascending=[True, False, True])
    )

Manual labels provided: 35
Valid labels found in final dataset: 12
Missing labels after upstream filtering: 23

Valid label counts:


,gold_y,count
0,1,6
1,0,6


Valid labels by department:


,gold_department,gold_y,count
0,Bioprocessing & Excipients - Cell Culture Media,1,1
2,Global Strategic Outsourcing & Devices I Devic...,1,3
1,Global Strategic Outsourcing & Devices I Devic...,0,4
4,Logistics,1,2
3,Logistics,0,1
5,Packaging Material,0,1


Matched gold contracts:


,contract_id,contract_number,contract_name,department,gold_y,gold_department
0,504361,504149,Stena Recycling A/S_Commercial Lease Agreement...,Bioprocessing and Excipients,1,Bioprocessing & Excipients - Cell Culture Media
18,4821,4822,Flextronics_Master_20130409_OA,Devices & Needles,1,Global Strategic Outsourcing & Devices I Devic...
19,557,557,Elos Medtech Tianjin Co. Ltd_Master_20091117_OA,Devices & Needles,1,Global Strategic Outsourcing & Devices I Devic...
20,565,565,Mesa Parts GmbH & Co. KG_Master_20051123_SA,Devices & Needles,1,Global Strategic Outsourcing & Devices I Devic...
27,1603,1603,Nemera La Verpilliere_Master_20181219_OA,Devices & Needles,0,Global Strategic Outsourcing & Devices I Devic...
26,1614,1614,Nipro Corporation_Master_20160226_OA,Devices & Needles,0,Global Strategic Outsourcing & Devices I Devic...
28,4507,4507,Nolato_Master_20170601_OA,Devices & Needles,0,Global Strategic Outsourcing & Devices I Devic...
24,483149,483178,RKT_Rodinger_Kunststoff_Technik_GmbH_Master_20...,Devices & Needles,0,Global Strategic Outsourcing & Devices I Devic...
4,192312,192325,SkyNRG_Stand-Alone_2021_Partnership Agreement ...,Logistics,1,Logistics
5,541883,541269,KUEHNE + NAGEL A/S_MASTER_2017_NNAS_Primary_LS...,Logistics,1,Logistics


Missing gold contracts:


,contract_number,gold_y,gold_department
1,612555,1,Bioprocessing & Excipients - Cell Culture Media
3,1877,0,Bioprocessing & Excipients - Cell Culture Media
2,346312,0,Bioprocessing & Excipients - Cell Culture Media
21,301059,1,Global Strategic Outsourcing & Devices I Devic...
22,352237,1,Global Strategic Outsourcing & Devices I Devic...
23,518137,1,Global Strategic Outsourcing & Devices I Devic...
25,25777,0,Global Strategic Outsourcing & Devices I Devic...
8,456272,1,Logistics
7,470427,1,Logistics
10,508089,1,Logistics


## Split feasibility warning

Stage 1 requires at least:
- one positive and one negative contract in the training split
- one positive and one negative contract in the test split

If too few negative contracts survive the upstream filters, a balanced gold split is impossible.
In that case, the correct fix is to obtain more valid negative examples from the domain expert and
verify that they exist in the final processed dataset before rerunning Stage 1.

In [156]:
n_valid_positive = int((df_gold_valid["gold_y"] == 1).sum())
n_valid_negative = int((df_gold_valid["gold_y"] == 0).sum())

print(f"Valid positive contracts: {n_valid_positive}")
print(f"Valid negative contracts: {n_valid_negative}")

if n_valid_negative < 2:
    print()
    print("WARNING:")
    print(
        "There are fewer than 2 valid negative gold contracts in the final dataset. "
        "A balanced train/test split is therefore not possible."
    )
    print(
        "Recommended action: collect additional negative examples, confirm that they survive "
        "the upstream filtering pipeline, and then update `gold_contract_map`."
    )
else:
    print()
    print("Gold split feasibility check passed: enough negatives exist to attempt a balanced split.")

Valid positive contracts: 6
Valid negative contracts: 6

Gold split feasibility check passed: enough negatives exist to attempt a balanced split.


## Join gold labels onto final contract table

In [157]:
df_contract_final_no_spend["contract_number"] = clean_id_like_column(df_contract_final_no_spend["contract_number"])
df_gold_manual["contract_number"] = clean_id_like_column(df_gold_manual["contract_number"])

df_contract_with_gold = df_contract_final_no_spend.merge(
    df_gold_manual,
    on="contract_number",
    how="left",
)

print("Joined shape:", df_contract_with_gold.shape)
print("Rows with gold_y:", df_contract_with_gold["gold_y"].notna().sum())
print(
    "Unique gold-labeled contracts in final table:",
    df_contract_with_gold.loc[df_contract_with_gold["gold_y"].notna(), "contract_number"].nunique(),
)

display(
    df_contract_with_gold[
        ["contract_id", "contract_number", "contract_name", "department", "gold_y", "gold_department"]
    ]
    .dropna(subset=["gold_y"])
    .drop_duplicates()
    .sort_values(["gold_department", "gold_y", "contract_number"], ascending=[True, False, True])
)

Joined shape: (9201, 149)
Rows with gold_y: 84
Unique gold-labeled contracts in final table: 12


,contract_id,contract_number,contract_name,department,gold_y,gold_department
5100,504361,504149,Stena Recycling A/S_Commercial Lease Agreement...,Bioprocessing and Excipients,1.0,Bioprocessing & Excipients - Cell Culture Media
3459,4821,4822,Flextronics_Master_20130409_OA,Devices & Needles,1.0,Global Strategic Outsourcing & Devices I Devic...
3080,557,557,Elos Medtech Tianjin Co. Ltd_Master_20091117_OA,Devices & Needles,1.0,Global Strategic Outsourcing & Devices I Devic...
3157,565,565,Mesa Parts GmbH & Co. KG_Master_20051123_SA,Devices & Needles,1.0,Global Strategic Outsourcing & Devices I Devic...
3271,1603,1603,Nemera La Verpilliere_Master_20181219_OA,Devices & Needles,0.0,Global Strategic Outsourcing & Devices I Devic...
3380,1614,1614,Nipro Corporation_Master_20160226_OA,Devices & Needles,0.0,Global Strategic Outsourcing & Devices I Devic...
477,4507,4507,Nolato_Master_20170601_OA,Devices & Needles,0.0,Global Strategic Outsourcing & Devices I Devic...
2932,483149,483178,RKT_Rodinger_Kunststoff_Technik_GmbH_Master_20...,Devices & Needles,0.0,Global Strategic Outsourcing & Devices I Devic...
425,192312,192325,SkyNRG_Stand-Alone_2021_Partnership Agreement ...,Logistics,1.0,Logistics
194,541883,541269,KUEHNE + NAGEL A/S_MASTER_2017_NNAS_Primary_LS...,Logistics,1.0,Logistics


In [158]:
df_contract_with_gold.head()

,contract_id,contract_number,contract_name,contract_status,contract_owner_cost_centre,terminated,term_type,start_date,expiration_date,supplier_id,...,LPI_Score,Customs,Infrastructure,International_Shipments,Logistics_Competence,Tracking_Tracing,Timeliness,PPI_Value,gold_y,gold_department
0,9675,9675,Bioreliance_Master_2018_MSA,published,7756.0,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,...,3.991859,3.918058,3.95873,3.530159,4.007843,4.176078,4.407843,84.4,NaN,NaN
1,9675,9675,Bioreliance_Master_2018_MSA,published,7756.0,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,...,3.991859,3.918058,3.95873,3.530159,4.007843,4.176078,4.407843,83.9,NaN,NaN
2,9675,9675,Bioreliance_Master_2018_MSA,published,7756.0,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,...,3.991859,3.918058,3.95873,3.530159,4.007843,4.176078,4.407843,79.9,NaN,NaN
3,9675,9675,Bioreliance_Master_2018_MSA,published,7756.0,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,...,3.991859,3.918058,3.95873,3.530159,4.007843,4.176078,4.407843,100.0,NaN,NaN
4,9675,9675,Bioreliance_Master_2018_MSA,published,7756.0,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,...,3.991859,3.918058,3.95873,3.530159,4.007843,4.176078,4.407843,147.0,NaN,NaN


In [159]:
df_gold_manual.head()

,contract_number,gold_y,gold_department
0,504149,1,Bioprocessing & Excipients - Cell Culture Media
1,612555,1,Bioprocessing & Excipients - Cell Culture Media
2,346312,0,Bioprocessing & Excipients - Cell Culture Media
3,1877,0,Bioprocessing & Excipients - Cell Culture Media
4,192325,1,Logistics


## Save outputs

Files saved:
- `contract_final_no_spend.csv`
- `gold_labels_manual.csv`
- `contract_with_gold.csv`

In [160]:
save_processed(df_contract_final_no_spend, "contract_final_no_spend.csv")
save_processed(df_gold_manual, "gold_labels_manual.csv")
save_processed(df_contract_with_gold, "contract_with_gold.csv")

print("Saved:")
print("- Data/processed/contract_final_no_spend.csv")
print("- Data/processed/gold_labels_manual.csv")
print("- Data/processed/contract_with_gold.csv")

Saved:
- Data/processed/contract_final_no_spend.csv
- Data/processed/gold_labels_manual.csv
- Data/processed/contract_with_gold.csv


Temporary random gold labels for Stage 2 pipeline testing

In [161]:
# ---------------------------------------------------------
# Unified temporary labels for Stage 2 pipeline testing
#
# Purpose:
# - sample temporary meta-train labels from non-Logistics departments
# - sample temporary meta-test labels from Logistics
# - combine both into one single Stage 2 test label table
# - always use contract_id as the primary key
# - save one unified Stage 2 label table for downstream Stage 2 prep
# ---------------------------------------------------------

import numpy as np
import pandas as pd

print("Building unified temporary labels for Stage 2 pipeline testing...")
print("Non-Logistics contracts will be used for meta-train.")
print("Logistics contracts will be used for meta-test.")
print("Primary key for Stage 2 labels: contract_id")

# Make sure identifiers are clean
df_contract_final_no_spend["contract_number"] = clean_id_like_column(
    df_contract_final_no_spend["contract_number"]
)
df_contract_final_no_spend["contract_id"] = clean_id_like_column(
    df_contract_final_no_spend["contract_id"]
)

# Build one-row-per-contract lookup table for sampling
df_contract_lookup = (
    df_contract_final_no_spend[
        ["contract_id", "contract_number", "department"]
    ]
    .drop_duplicates(subset=["contract_id"])
    .copy()
)

print("\nUnique contracts available for sampling:", len(df_contract_lookup))

# =========================================================
# 1. Temporary meta-train labels from NON-LOGISTICS
# =========================================================
np.random.seed(42)

df_meta_train_pool = df_contract_lookup[
    df_contract_lookup["department"] != "Logistics"
].copy()

meta_train_contract_ids = df_meta_train_pool["contract_id"].dropna().astype(str).unique()

n_meta_train_yes = 15
n_meta_train_no = 15

meta_train_yes_ids = np.random.choice(
    meta_train_contract_ids,
    size=n_meta_train_yes,
    replace=False
)

remaining_meta_train_ids = list(set(meta_train_contract_ids) - set(meta_train_yes_ids))

meta_train_no_ids = np.random.choice(
    remaining_meta_train_ids,
    size=n_meta_train_no,
    replace=False
)

df_gold_meta_train = pd.DataFrame({
    "contract_id": list(meta_train_yes_ids) + list(meta_train_no_ids),
    "gold_y": [1] * len(meta_train_yes_ids) + [0] * len(meta_train_no_ids),
    "gold_department": ["Random_Meta_Train"] * (len(meta_train_yes_ids) + len(meta_train_no_ids)),
})

# Attach contract_number for readability only
df_gold_meta_train = df_gold_meta_train.merge(
    df_contract_lookup[["contract_id", "contract_number"]],
    on="contract_id",
    how="left",
)

print("\nTemporary non-Logistics meta-train label table shape:", df_gold_meta_train.shape)

# =========================================================
# 2. Temporary meta-test labels from LOGISTICS
# =========================================================
np.random.seed(123)

df_logistics_pool = df_contract_lookup[
    df_contract_lookup["department"] == "Logistics"
].copy()

logistics_contract_ids = df_logistics_pool["contract_id"].dropna().astype(str).unique()

n_logistics_yes = 5
n_logistics_no = 5

logistics_yes_ids = np.random.choice(
    logistics_contract_ids,
    size=n_logistics_yes,
    replace=False
)

remaining_logistics_ids = list(set(logistics_contract_ids) - set(logistics_yes_ids))

logistics_no_ids = np.random.choice(
    remaining_logistics_ids,
    size=n_logistics_no,
    replace=False
)

df_gold_logistics = pd.DataFrame({
    "contract_id": list(logistics_yes_ids) + list(logistics_no_ids),
    "gold_y": [1] * len(logistics_yes_ids) + [0] * len(logistics_no_ids),
    "gold_department": ["Logistics"] * (len(logistics_yes_ids) + len(logistics_no_ids)),
})

# Attach contract_number for readability only
df_gold_logistics = df_gold_logistics.merge(
    df_contract_lookup[["contract_id", "contract_number"]],
    on="contract_id",
    how="left",
)

print("\nTemporary Logistics meta-test label table shape:", df_gold_logistics.shape)

# =========================================================
# 3. Combine both label tables into ONE Stage 2 label table
# =========================================================
df_gold_stage2_test = pd.concat(
    [df_gold_meta_train, df_gold_logistics],
    ignore_index=True
).drop_duplicates(subset=["contract_id"]).copy()

df_gold_stage2_test["contract_id"] = clean_id_like_column(df_gold_stage2_test["contract_id"])
df_gold_stage2_test["contract_number"] = clean_id_like_column(df_gold_stage2_test["contract_number"])

print("\nUnified Stage 2 temporary label table shape:", df_gold_stage2_test.shape)
print("Unique labeled contracts:", df_gold_stage2_test["contract_id"].nunique())

# =========================================================
# 4. Debug join onto full contract-year table (optional)
# =========================================================
df_contract_with_gold_stage2_test = df_contract_final_no_spend.merge(
    df_gold_stage2_test[["contract_id", "gold_y", "gold_department"]],
    on="contract_id",
    how="left",
)

print("\nUnified Stage 2 test dataset shape:", df_contract_with_gold_stage2_test.shape)
print("Rows with temporary gold_y:", df_contract_with_gold_stage2_test["gold_y"].notna().sum())
print(
    "Unique labeled contracts:",
    df_contract_with_gold_stage2_test.loc[
        df_contract_with_gold_stage2_test["gold_y"].notna(),
        "contract_id"
    ].nunique()
)

print("\nDepartments represented in unified Stage 2 temporary labels:")
display(
    df_contract_with_gold_stage2_test[
        ["contract_id", "department", "gold_y"]
    ]
    .dropna(subset=["gold_y"])
    .drop_duplicates()
    ["department"]
    .value_counts()
)

print("\nSample of labeled rows in unified Stage 2 test dataset:")
display(
    df_contract_with_gold_stage2_test[
        ["contract_id", "contract_number", "contract_name", "observation_year", "department", "gold_y", "gold_department"]
    ]
    .dropna(subset=["gold_y"])
    .drop_duplicates()
    .sort_values(["department", "contract_id", "observation_year"])
    .head(50)
)


Building unified temporary labels for Stage 2 pipeline testing...
Non-Logistics contracts will be used for meta-train.
Logistics contracts will be used for meta-test.
Primary key for Stage 2 labels: contract_id

Unique contracts available for sampling: 2209

Temporary non-Logistics meta-train label table shape: (30, 4)

Temporary Logistics meta-test label table shape: (10, 4)

Unified Stage 2 temporary label table shape: (40, 4)
Unique labeled contracts: 40

Unified Stage 2 test dataset shape: (9201, 149)
Rows with temporary gold_y: 184
Unique labeled contracts: 40

Departments represented in unified Stage 2 temporary labels:


department
Logistics                           10
Devices & Needles                   10
Drug Substance Outsourcing           6
Bioprocessing and Excipients         4
Drug Product Outsourcing             4
Raw Materials & Energy               3
Alliance, Acquisitions & PPM CoE     1
Packaging Material                   1
Bioprocessing & Raw Materials        1
Name: count, dtype: int64


Sample of labeled rows in unified Stage 2 test dataset:


,contract_id,contract_number,contract_name,observation_year,department,gold_y,gold_department
2131,7068,7068,Cathay_master_2008_SA,2015,"Alliance, Acquisitions & PPM CoE",1.0,Random_Meta_Train
2132,7068,7068,Cathay_master_2008_SA,2016,"Alliance, Acquisitions & PPM CoE",1.0,Random_Meta_Train
2133,7068,7068,Cathay_master_2008_SA,2017,"Alliance, Acquisitions & PPM CoE",1.0,Random_Meta_Train
2134,7068,7068,Cathay_master_2008_SA,2018,"Alliance, Acquisitions & PPM CoE",1.0,Random_Meta_Train
2135,7068,7068,Cathay_master_2008_SA,2019,"Alliance, Acquisitions & PPM CoE",1.0,Random_Meta_Train
2136,7068,7068,Cathay_master_2008_SA,2020,"Alliance, Acquisitions & PPM CoE",1.0,Random_Meta_Train
2137,7068,7068,Cathay_master_2008_SA,2021,"Alliance, Acquisitions & PPM CoE",1.0,Random_Meta_Train
2138,7068,7068,Cathay_master_2008_SA,2022,"Alliance, Acquisitions & PPM CoE",1.0,Random_Meta_Train
2139,7068,7068,Cathay_master_2008_SA,2023,"Alliance, Acquisitions & PPM CoE",1.0,Random_Meta_Train
2140,7068,7068,Cathay_master_2008_SA,2024,"Alliance, Acquisitions & PPM CoE",1.0,Random_Meta_Train


In [162]:
# =========================================================
# 5. Save only the useful final Stage 2 testing artifacts
# =========================================================
save_processed(
    df_gold_stage2_test[["contract_id", "contract_number", "gold_y", "gold_department"]],
    "gold_labels_stage2_test.csv"
)

# Optional debug artifact only
save_processed(
    df_contract_with_gold_stage2_test,
    "contract_with_gold_stage2_test.csv"
)

print("\nSaved:")
print("- Data/processed/gold_labels_stage2_test.csv")
print("- Data/processed/contract_with_gold_stage2_test.csv")


Saved:
- Data/processed/gold_labels_stage2_test.csv
- Data/processed/contract_with_gold_stage2_test.csv
